# Wrapper Feature Selection — Example Notebook

Demonstrates **`WrapperSelector`** on the SCANB gene-expression dataset.

| Task | Target | Type | Algorithm | SU Mode |
|------|--------|------|-----------|----------|
| **A** | `is_lumA` | classification | IWSS-MB | Mode A — precomputed SU matrix (with MI interim save) |
| **B** | `Lympho` | regression | IWSS-MB | Mode A — precomputed SU matrix |
| **C** | `is_lumA` | classification | all four variants | Mode B — lazy incremental fill |

**Features demonstrated:**
- `ScoreMatrix` with `method="su"` — replaces `ScoreMatrix`
- Mode A precomputation (CSV/Parquet load/save) and Mode B lazy `ScoreCalculator` fill
- `cv_folds` / `cv_min_folds` dual-condition acceptance
- `PATIENCE` stopping (global, default `None`)
- `test_eval_every_k` validation evaluation
- `WrapperSelectionResult` timing breakdown
- `SelectionResult.compare_results` for multi-algorithm comparison
- `SelectionResult.compare_with` for pairwise rank-aligned feature diff
- `SelectionResult.summary_report` for multi-result feature summary
- `plot_vs_random_baseline` for classification and regression

## 0 — Imports & paths

In [ ]:
import sys, os

# Make sure the package root is on the path when running from feature_selection/src/
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from feature_selection.src.preprocessing import train_test_val_split
from feature_selection.src.score_matrix import ScoreMatrix
from feature_selection.src.wrapper.MarkovBlanketWrapper import WrapperSelector
from feature_selection.src.results import SelectionResult, WrapperSelectionResult
from feature_selection.src.LR_random_baseline import plot_performance_with_stats as clf_baseline_plot
from feature_selection.src.LR_regression_baseline import plot_performance_with_stats as reg_baseline_plot

# ── Paths (relative to notebook location: feature_selection/src/) ──────────
DATA_PATH  = '../data/SCANB.csv'
LABEL_PATH = '../data/sampleinfo_SCANB_t.csv'

# Precomputed SU matrix — single file shared across all tasks on the same feature set
# Feature-pair SU values are target-independent; only the target column differs.
SU_PATH = '../data/su_matrix.csv'   # shared SU matrix (features + all targets)

# ── Globals ────────────────────────────────────────────────────────────────
RANDOM_SEED     = 2
N_FEATURES      = 20    # features to select
TEST_EVAL_EVERY = 2     # run test/val evaluation every k selections
N_RUNS          = 10    # random baseline runs
CV_FOLDS        = 5
CV_MIN_FOLDS    = 3
PATIENCE        = None  # stop after N consecutive non-improvements; None = no limit

## 1 — Load data & build train / validation splits

In [ ]:
gene_expression_df = pd.read_csv(DATA_PATH)
sampleinfo_df      = pd.read_csv(LABEL_PATH)

# Binary LumA label
sampleinfo_df['is_lumA'] = (sampleinfo_df['PAM50'] == 'LumA').astype(int)

# 70% train / 10% val / 20% test
train_si, val_si, _ = train_test_val_split(
    sampleinfo_df, random_seed=RANDOM_SEED, train_pcnt=0.7, val_pct=0.1
)

train_labels_df = train_si[['samplename', 'is_lumA']].reset_index(drop=True)
val_labels_df   = val_si[['samplename', 'is_lumA']].reset_index(drop=True)

print(f'Train samples : {len(train_labels_df)}')
print(f'Val   samples : {len(val_labels_df)}')
print(f"LumA prevalence (train): {train_labels_df['is_lumA'].mean():.2%}")

# Transpose gene-expression (genes × samples) → (samples × genes)
ge_proc = gene_expression_df.copy().fillna(0).set_index('Unnamed: 0').T
ge_proc.index.name = 'samplename'

X_train = ge_proc.loc[train_labels_df['samplename']]
X_val   = ge_proc.loc[val_labels_df['samplename']]

print(f'\nX_train shape : {X_train.shape}')
print(f'X_val   shape : {X_val.shape}')

## 2 — Build target series

In [ ]:
# Classification target: is_lumA
y_lumA = (
    train_labels_df.set_index('samplename')['is_lumA']
    .reindex(X_train.index)
)  # Series.name == 'is_lumA'

y_lumA_val = (
    val_labels_df.set_index('samplename')['is_lumA']
    .reindex(X_val.index)
)

# Regression target: Lympho
y_lympho = (
    sampleinfo_df.set_index('samplename')['Lympho']
    .reindex(X_train.index)
)  # Series.name == 'Lympho'

y_lympho_val = (
    sampleinfo_df.set_index('samplename')['Lympho']
    .reindex(X_val.index)
)

print(f'y_lumA   name={y_lumA.name!r}    dtype={y_lumA.dtype}   nunique={y_lumA.nunique()}')
print(f'y_lympho name={y_lympho.name!r}  dtype={y_lympho.dtype}   nunique={y_lympho.nunique()}')

---

## Precomputation — SU matrix (Mode A)

**Run once** to compute and save the full Symmetric Uncertainty matrix.
On subsequent runs the file is loaded directly — no recomputation.

The new `ScoreMatrix` class with `method='su'` replaces the old
precomputation classes.  It uses `ScoreCalculator`
internally with the fully vectorised `np.bincount`-based SU engine.

> **Single shared matrix**  
> Feature-pair SU values are identical regardless of which target is used, so
> there is no reason to recompute them for every task.  We build **one** matrix
> that includes both target columns (`is_lumA` and `Lympho`).


### Precompute shared SU matrix (both targets: `is_lumA` and `Lympho`)

Uses `ScoreMatrix(method='su')` — the new vectorised SU engine.


In [ ]:
if not os.path.exists(SU_PATH):
    print('Computing shared SU matrix for both targets...')
    print(f'  SU matrix → {SU_PATH}')
    ScoreMatrix(
        X=X_train.fillna(0).values,
        feature_names=list(X_train.columns),
        target_names=[str(y_lumA.name), str(y_lympho.name)],
        target_data=pd.concat([y_lumA, y_lympho], axis=1).fillna(0).values,
        method='su',
        filepath=SU_PATH,
        file_format='csv',
    ).precompute()
else:
    print(f'SU matrix already exists at {SU_PATH} — skipping computation.')

# Inspect the saved SU matrix
if os.path.exists(SU_PATH):
    su_shared = pd.read_csv(SU_PATH, index_col=0)
    print(f'\nShared SU matrix shape: {su_shared.shape}')
    if 'is_lumA' in su_shared.columns:
        print('Top-5 SU values with is_lumA target:')
        print(su_shared['is_lumA'].drop('is_lumA', errors='ignore').sort_values(ascending=False).head(5).to_string())
    if 'Lympho' in su_shared.columns:
        print('\nTop-5 SU values with Lympho target:')
        print(su_shared['Lympho'].drop('Lympho', errors='ignore').sort_values(ascending=False).head(5).to_string())


---

## Task A — Classification (is_lumA), IWSS-MB, Mode A

### A.1 — Run IWSS-MB selector

In [ ]:
ws_clf = WrapperSelector(
    X_train=X_train,
    y_train=y_lumA,
    X_val=X_val,
    y_val=y_lumA_val,
    use_su_ranking=True,          # IWSS: rank by SU(f, C) descending
    use_mb_pruning=True,          # MB: prune redundant candidates after each selection
    cv_folds=CV_FOLDS,
    cv_min_folds=CV_MIN_FOLDS,
    patience=PATIENCE,
    su_filepath=SU_PATH,          # Mode A: shared SU matrix (feature-pairs computed once)
    random_seed=RANDOM_SEED,
)

result_clf: WrapperSelectionResult = ws_clf.run(
    n_features_to_select=N_FEATURES,
    test_eval_every_k=TEST_EVAL_EVERY,
)

print()
print(result_clf)

### A.2 — Timing breakdown

In [ ]:
print('Timing breakdown:')
print(f'  Total selection time   : {result_clf.selection_time_seconds:.3f}s')
print(f'    └─ Filter init time  : {result_clf.filter_time_seconds:.3f}s')
print(f'    └─ Evaluator CV time : {result_clf.classifier_time_seconds:.3f}s')
print(f'  Test eval time (val)   : {result_clf.test_evaluation_time_seconds:.3f}s')
print(f'\nWrapper CV evaluations   : {result_clf.n_wrapper_evaluations}')
print(f'Candidates pruned (MB)   : {result_clf.n_candidates_pruned}')
print(f'Evaluations skipped (MB) : {result_clf.n_evaluations_skipped}')
print(f'Stopping reason          : {result_clf.stopping_reason}')

### A.3 — Selected features

In [ ]:
print(f'Classification — {len(result_clf.selected_features)} features selected (in order):')
for i, f in enumerate(result_clf.selected_features, 1):
    print(f'  {i:2d}. {f}')

# Inspect first history entry — includes 'step' key
if result_clf.performance_history:
    print('\nFirst performance history entry:')
    display(pd.Series({
        k: v for k, v in result_clf.performance_history[0].items()
        if k not in ('macro avg', 'weighted avg')
    }))

### A.4 — LR random baseline

In [ ]:
# N_values = number of features selected at each evaluated checkpoint (NOT a loop index)
checkpoints_clf = [entry['step'] for entry in result_clf.performance_history]

baseline_summary_clf = clf_baseline_plot(
    gene_expression_df=gene_expression_df,
    train_labels_df=train_labels_df,
    val_labels_df=val_labels_df,
    N_values=checkpoints_clf,
    random_seed=RANDOM_SEED,
    num_runs=N_RUNS,
    return_summary=True,
)

### A.5 — Comparison plot

In [ ]:
result_clf.plot_vs_random_baseline(
    baseline_summary=baseline_summary_clf,
    title_suffix=' — SCANB IWSS-MB (is_lumA)',
)

---

## Task B — Regression (Lympho), IWSS-MB, Mode A

### B.1 — Run IWSS-MB selector

In [ ]:
ws_reg = WrapperSelector(
    X_train=X_train,
    y_train=y_lympho,
    X_val=X_val,
    y_val=y_lympho_val,
    use_su_ranking=True,
    use_mb_pruning=True,
    cv_folds=CV_FOLDS,
    cv_min_folds=CV_MIN_FOLDS,
    patience=PATIENCE,
    su_filepath=SU_PATH,          # same shared SU matrix — Lympho column already present
    random_seed=RANDOM_SEED,
)

result_reg: WrapperSelectionResult = ws_reg.run(
    n_features_to_select=N_FEATURES,
    test_eval_every_k=TEST_EVAL_EVERY,
)

print()
print(result_reg)

### B.2 — Selected features

In [ ]:
print(f'Regression — {len(result_reg.selected_features)} features selected (in order):')
for i, f in enumerate(result_reg.selected_features, 1):
    print(f'  {i:2d}. {f}')

### B.3 — LinReg random baseline

In [ ]:
# N_values = number of features selected at each evaluated checkpoint
checkpoints_reg = [entry['step'] for entry in result_reg.performance_history]

baseline_summary_reg = reg_baseline_plot(
    X_train=X_train,
    y_train=y_lympho,
    X_val=X_val,
    y_val=y_lympho_val,
    N_values=checkpoints_reg,
    random_seed=RANDOM_SEED,
    num_runs=N_RUNS,
    return_summary=True,
)

### B.4 — Comparison plot

In [ ]:
result_reg.plot_vs_random_baseline(
    baseline_summary=baseline_summary_reg,
    title_suffix=' — SCANB IWSS-MB (Lympho)',
)

---

## Task C — All four algorithm variants compared (is_lumA, Mode B)

Run all four `WrapperSelector` variants without a precomputed SU file (Mode B —
lazy incremental column-fill) and compare their performance histories.

| Variant | `use_su_ranking` | `use_mb_pruning` |
|---------|-----------------|------------------|
| SFS     | False           | False            |
| SFS-MB  | False           | True             |
| IWSS    | True            | False            |
| IWSS-MB | True            | True             |

### C.1 — Run all four variants

In [ ]:
N_COMPARE = 10   # fewer features to keep SFS runtime manageable

variants = [
    ('SFS',     False, False),
    ('SFS-MB',  False, True),
    ('IWSS',    True,  False),
    ('IWSS-MB', True,  True),
]

results_C = {}   # label → WrapperSelectionResult

for label, use_su, use_mb in variants:
    print(f'\n=== {label} (use_su_ranking={use_su}, use_mb_pruning={use_mb}) ===')
    ws = WrapperSelector(
        X_train=X_train,
        y_train=y_lumA,
        X_val=X_val,
        y_val=y_lumA_val,
        use_su_ranking=use_su,
        use_mb_pruning=use_mb,
        cv_folds=CV_FOLDS,
        cv_min_folds=CV_MIN_FOLDS,
        patience=PATIENCE,
        # su_filepath=None → Mode B lazy fill
        random_seed=RANDOM_SEED,
    )
    result = ws.run(
        n_features_to_select=N_COMPARE,
        test_eval_every_k=1,
    )
    results_C[label] = result
    print(f'  Selected: {result.selected_features}')
    print(f'  Stopping: {result.stopping_reason}')
    print(f'  CV evals: {result.n_wrapper_evaluations}  '
          f'Pruned: {result.n_candidates_pruned}  '
          f'Time: {result.selection_time_seconds:.3f}s')

### C.2 — Timing and efficiency comparison table

In [ ]:
timing_rows = []
for label, res in results_C.items():
    timing_rows.append({
        'variant': label,
        'features_selected': len(res.selected_features),
        'stopping_reason': res.stopping_reason,
        'total_time_s': round(res.selection_time_seconds, 4),
        'filter_time_s': round(res.filter_time_seconds, 4),
        'clf_time_s': round(res.classifier_time_seconds, 4),
        'cv_evaluations': res.n_wrapper_evaluations,
        'pruned': res.n_candidates_pruned,
        'skipped_evals': res.n_evaluations_skipped,
    })

timing_df = pd.DataFrame(timing_rows).set_index('variant')
print('Algorithm variant comparison:')
display(timing_df)

### C.3 — Evaluator-call count and timing bar charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

variants_order = list(results_C.keys())
eval_counts  = [results_C[v].n_wrapper_evaluations for v in variants_order]
total_times  = [results_C[v].selection_time_seconds for v in variants_order]

ax = axes[0]
bars = ax.bar(variants_order, eval_counts, color='steelblue', alpha=0.85)
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_title('Evaluator calls per variant', fontsize=12)
ax.set_ylabel('CV evaluations')
ax.grid(True, axis='y', alpha=0.4)

ax = axes[1]
bars = ax.bar(variants_order, total_times, color='darkorange', alpha=0.85)
ax.bar_label(bars, fmt='{:.2f}s', padding=3, fontsize=9)
ax.set_title('Total selection time per variant', fontsize=12)
ax.set_ylabel('Time (s)')
ax.grid(True, axis='y', alpha=0.4)

fig.suptitle(
    f'Task C — All variants, Mode B — SCANB (is_lumA, {N_COMPARE} features)',
    fontsize=13, fontweight='bold',
)
plt.tight_layout()
plt.show()

### C.4 — Performance comparison plot

In [ ]:
# Only compare results that have performance history entries
valid_results = {k: v for k, v in results_C.items() if v.performance_history}

if valid_results:
    SelectionResult.compare_results(
        results=list(valid_results.values()),
        labels=list(valid_results.keys()),
        baseline_summary=baseline_summary_clf,
        title_suffix=' — SCANB (is_lumA, Mode B)',
    )
else:
    print('No results with performance history to compare.')

### C.5 — Feature diff: IWSS vs IWSS-MB

In [ ]:
if 'IWSS' in results_C and 'IWSS-MB' in results_C:
    results_C['IWSS'].compare_with(
        results_C['IWSS-MB'],
        self_label='IWSS',
        other_label='IWSS-MB',
    )
else:
    print('IWSS or IWSS-MB result not available.')

### C.6 — Summary report: all four variants

In [ ]:
summary_df = SelectionResult.summary_report(
    results=list(results_C.values()),
    labels=list(results_C.keys()),
)

---

## Summary — Timing across all tasks

In [ ]:
print('=== Task A — Classification IWSS-MB (is_lumA, Mode A) ===')
print(f'  Features selected    : {len(result_clf.selected_features)}')
print(f'  Total time           : {result_clf.selection_time_seconds:.3f}s')
print(f'    └─ Filter init     : {result_clf.filter_time_seconds:.3f}s')
print(f'    └─ Evaluator CV    : {result_clf.classifier_time_seconds:.3f}s')
print(f'  Test eval time (val) : {result_clf.test_evaluation_time_seconds:.3f}s')
print(f'  CV evaluations       : {result_clf.n_wrapper_evaluations}')
print(f'  Candidates pruned    : {result_clf.n_candidates_pruned}')
print(f'  Stopping reason      : {result_clf.stopping_reason}')

print()
print('=== Task B — Regression IWSS-MB (Lympho, Mode A) ===')
print(f'  Features selected    : {len(result_reg.selected_features)}')
print(f'  Total time           : {result_reg.selection_time_seconds:.3f}s')
print(f'    └─ Filter init     : {result_reg.filter_time_seconds:.3f}s')
print(f'    └─ Evaluator CV    : {result_reg.classifier_time_seconds:.3f}s')
print(f'  Test eval time (val) : {result_reg.test_evaluation_time_seconds:.3f}s')
print(f'  CV evaluations       : {result_reg.n_wrapper_evaluations}')
print(f'  Candidates pruned    : {result_reg.n_candidates_pruned}')
print(f'  Stopping reason      : {result_reg.stopping_reason}')

print()
print('=== Task C — All four variants (is_lumA, Mode B) ===')
display(timing_df)